# Formula 1 Pipeline Run:

## 1) Import Configuration and Common Functions:

In [0]:
%run ./common/configuration


In [0]:
%run ./common/functions

# 2) Test connection with Azure Data Lake Storage

In [0]:
# dbutils.notebook.run(
#     "./ingest/test_access_adls_using_access_keys",
#     600,
#     {"dbfs_location": dbfs_location, "demo_location": demo_location},
# )


# 3) Import Raw Data (Uncomment this code to import data from Jolpica F1 API)

In [0]:
# dbutils.notebook.run(
#     "./ingest/import_data",
#     3600,
#     {"raw_folder_path": raw_folder_path, "processed_folder_path": processed_folder_path},
# )


# 4) Convert all Imported Data to CSV (and drop any tables ):

In [0]:
# dbutils.notebook.run(
#     "./ingest/convert_to_csv",
#     3600,
#     {"raw_folder_path": raw_folder_path, "processed_folder_path": processed_folder_path},
# )



# 5) Process Converted CSVs:

In [0]:
dbutils.notebook.run(
    "./process/process_all_files",
    600,
)

# 6) List all tables along with rows and column count"

In [0]:
databases_to_check = ["f1_processed", "f1_presentation"]
summary_rows = []

for db_name in databases_to_check:
    try:
        table_names = [row.tableName for row in spark.sql(f"SHOW TABLES IN {db_name}").collect()]
    except Exception as e:
        print(f"Could not list tables in {db_name}: {e}")
        continue

    for table_name in table_names:
        try:
            df = spark.table(f"{db_name}.{table_name}")
            summary_rows.append((db_name, table_name, df.count(), len(df.columns)))
        except Exception as e:
            summary_rows.append((db_name, table_name, None, None))
            print(f"Could not read {db_name}.{table_name}: {e}")

summary_df = spark.createDataFrame(summary_rows, ["database", "table_name", "row_count", "column_count"])
display(summary_df)